# Sinh dữ liệu `can_nav_pnp` trên Colab

Chạy `scripts/datagen/can_nav_pnp/run_blocks.py` trên Colab. Seed được chia thành **khối nhỏ** (mặc định 8 episode/khối).
Khối nào xong được chép ngay về Drive. Runtime bị thu hồi thì chạy lại notebook: các khối đã có trên Drive được bỏ qua,
chỉ mất các khối đang chạy dở.

**Tốc độ phụ thuộc CPU và RAM, không phụ thuộc GPU.** Mô phỏng MuJoCo và SONIC (onnxruntime) chạy trên CPU, mỗi tiến trình
tốn khoảng 2,5 GB RAM. GPU chỉ giúp phần render video.
- **CPU High-RAM** (khoảng 8 vCPU / 51 GB, rất ít compute unit): lựa chọn kinh tế nhất. Có thể mở vài phiên song song.
- **TPU v5e-1** (khoảng 24 vCPU / 48 GB): nhanh hơn trong một phiên.
- **GPU L4** (khoảng 12 vCPU / 53 GB): render video ngay bằng EGL trên GPU, dữ liệu ra là dùng được luôn. Tốn nhiều compute
  unit hơn CPU High-RAM.
- `RENDER = "auto"`: có GPU thì render bằng EGL; không có GPU thì **không render** (chỉ mô phỏng, render sau ở máy local bằng
  `render_views.py`). OSMesa (render bằng CPU) quá chậm khi nhiều tiến trình chạy cùng lúc, chỉ dùng khi đặt `RENDER = "osmesa"`.

### Chuẩn bị (một lần)
1. Trên máy local, tạo gói: `.venv-datagen/bin/python scripts/datagen/can_nav_pnp/pack_colab.py`
   (ghi ra `.data/colab/can_nav_datagen.zip`, khoảng 850 MB).
2. Tải file zip lên **My Drive/can_nav/** của tài khoản chạy Colab.

### Nhiều phiên cùng lúc
Mỗi phiên đặt một `SEED_BASE` **khác nhau**, ví dụ 100000, 300000, 500000. Mọi phiên ghi chung vào `MyDrive/can_nav/blocks`
và đếm chung `TARGET`, nên số episode tổng chỉ vượt `TARGET` chừng vài khối. Không dùng seed từ 1 000 000 trở lên (dành cho
đánh giá) hoặc dưới 50 000 (máy local đã dùng).

### Sau khi xong (trên máy local)
```bash
# tải thư mục MyDrive/can_nav/blocks về .data/can_nav_blocks, rồi:
python scripts/datagen/can_nav_pnp/merge_shards.py --out .data/can_nav_500 .data/can_nav_s{0,1,2,3,4} .data/can_nav_blocks/blk_*
python scripts/datagen/can_nav_pnp/render_views.py .data/can_nav_500 --views ego --jobs 4   # chỉ cần khi RENDER = "none"
```

In [ ]:
# [1] Cấu hình
SEED_BASE = 100000        # mỗi phiên Colab một giá trị riêng (100000, 300000, 500000, ...)
TARGET = 340              # tổng số episode muốn có trong DRIVE_BLOCKS (đếm chung mọi phiên)
LEVEL = 1                 # độ dao động của cảnh (generate.py --level; 1 = biên độ x1,5)
BLOCK_SIZE = 8            # episode mỗi khối
RENDER = "auto"           # "auto": runtime GPU (L4) render bằng EGL; runtime CPU không render (render sau ở local bằng
                          # render_views.py). Đặt tay: "egl" / "none" / "osmesa" (render bằng CPU, rất chậm)
WORKERS = "auto"          # số tiến trình; "auto" tính theo CPU và RAM

DRIVE_DIR = "/content/drive/MyDrive/can_nav"
ZIP_NAME = "can_nav_datagen.zip"
DRIVE_BLOCKS = f"{DRIVE_DIR}/blocks"

In [ ]:
# [2] Drive, phần cứng, số tiến trình
import os, shutil, subprocess
from google.colab import drive
drive.mount("/content/drive")
assert os.path.exists(f"{DRIVE_DIR}/{ZIP_NAME}"), f"Không thấy {DRIVE_DIR}/{ZIP_NAME}: tải file zip của pack_colab.py lên Drive."

ncpu = len(os.sched_getaffinity(0))
avail_gb = int(open("/proc/meminfo").read().split("MemAvailable:")[1].split()[0]) / 2**20
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
render = RENDER if RENDER != "auto" else ("egl" if has_gpu else "none")
workers = WORKERS if WORKERS != "auto" else max(1, min(ncpu - 1, int((avail_gb - 3) / 2.8)))
print(f"CPU: {ncpu} | RAM trống: {avail_gb:.0f} GB | GPU: {has_gpu} -> render {render}, {workers} tiến trình")
print(subprocess.run(["bash", "-c", "lscpu | grep 'Model name'; free -h | head -2"], capture_output=True, text=True).stdout)

In [ ]:
%%bash -s "$DRIVE_DIR" "$ZIP_NAME"
# [3] Giải nén và cài môi trường: Python 3.11 + đúng phiên bản thư viện như máy local (uv, khoảng 1-2 phút)
set -euo pipefail
cd /content
[ -d can_nav/psi0 ] || (mkdir -p can_nav && unzip -q "$1/$2" -d can_nav)
cat can_nav/psi0/PACK_INFO.json | head -5
export PATH="$HOME/.local/bin:$PATH"
command -v uv || curl -LsSf https://astral.sh/uv/install.sh | sh
[ -x /content/venv/bin/python ] || uv venv -q --python 3.11 /content/venv
uv pip install -q --python /content/venv/bin/python -r can_nav/psi0/requirements_datagen.txt
# OpenGL không màn hình: OSMesa (CPU) và EGL (GPU)
(dpkg -s libosmesa6 >/dev/null 2>&1 && dpkg -s libegl1 >/dev/null 2>&1) || \
  (apt-get -qq update && apt-get -qq install -y libosmesa6 libegl1 libgl1 > /dev/null)
# EGL phải chạy trên GPU NVIDIA: không có file vendor này, libglvnd rơi về Mesa llvmpipe (render bằng CPU, chậm khoảng 40 lần)
if command -v nvidia-smi >/dev/null && nvidia-smi >/dev/null 2>&1; then
  mkdir -p /usr/share/glvnd/egl_vendor.d
  [ -f /usr/share/glvnd/egl_vendor.d/10_nvidia.json ] || \
    echo '{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}' > /usr/share/glvnd/egl_vendor.d/10_nvidia.json
fi
/content/venv/bin/python -c "import mujoco, onnxruntime, numpy; print('mujoco', mujoco.__version__, '| onnxruntime', onnxruntime.__version__, '| numpy', numpy.__version__)"

In [ ]:
# [4] Kiểm tra render và đo tốc độ: render 30 khung camera đầu
import os, subprocess
env = {**os.environ, "SONIC_VARIANT": "sonic_v1_1"}
if render != "none":
    env.update(MUJOCO_GL=render, PYOPENGL_PLATFORM=render)
os.environ.update({k: env[k] for k in ("SONIC_VARIANT", "MUJOCO_GL", "PYOPENGL_PLATFORM") if k in env})
test = r"""
import sys, time; sys.path.insert(0, "/content/can_nav/psi0/scripts/datagen/can_nav_pnp")
import mujoco, numpy as np
from OpenGL import GL
from scene import sample_params, build_model, IMG_H, IMG_W
m = build_model(sample_params(1)); d = mujoco.MjData(m); mujoco.mj_forward(m, d)
r = mujoco.Renderer(m, IMG_H, IMG_W); r.update_scene(d, camera="head_camera"); img = r.render()
gpu = GL.glGetString(GL.GL_RENDERER).decode(); t = time.time()
for _ in range(30): r.update_scene(d, camera="head_camera"); r.render()
noise = float(np.abs(np.diff(img.astype(np.int16), axis=1)).mean())
r.close()
print(f"render bằng: {gpu}, {(time.time() - t) / 30 * 1000:.0f} ms/khung 640x480, độ nhiễu ảnh {noise:.1f}")
assert noise < 20, "ảnh render bị nhiễu: render hỏng"
"""
if render != "none":
    res = subprocess.run(["/content/venv/bin/python", "-c", test], env=env, capture_output=True, text=True)
    print(res.stdout + res.stderr[-2000:])
    assert res.returncode == 0, "render thử thất bại: xem lỗi ở trên"
    if render == "egl":
        assert "llvmpipe" not in res.stdout.lower(), ("EGL đang render bằng CPU (llvmpipe), rất chậm: kiểm tra runtime có GPU, "
                                                      "chạy lại cell [3], hoặc khởi động lại runtime rồi chạy lại từ đầu")
else:
    print("RENDER = none: bỏ qua (video render sau ở local bằng render_views.py)")

In [ ]:
# [5] Sinh dữ liệu. Cell này chạy cho tới khi đủ TARGET và in tiến độ mỗi phút: để cell chạy suốt (Colab thu hồi
# runtime không có cell nào chạy). Dừng cell (■) thì các khối đang chạy vẫn tiếp tục; chạy lại cell để theo dõi tiếp.
import os, re, subprocess, time, glob, json
from datetime import datetime, timedelta, timezone
os.makedirs("/content/work", exist_ok=True)
LOG = "/content/work/run_blocks.log"
cmd = ["/content/venv/bin/python", "/content/can_nav/psi0/scripts/datagen/can_nav_pnp/run_blocks.py",
       "--out-dir", DRIVE_BLOCKS, "--work-dir", "/content/work", "--seed-base", str(SEED_BASE), "--target", str(TARGET),
       "--workers", str(workers), "--block-size", str(BLOCK_SIZE), "--level", str(LEVEL), "--threads", "1"]
if render == "none":
    cmd.append("--no-render")
if "run_proc" not in globals() or run_proc.poll() is not None:
    run_proc = subprocess.Popen(cmd, stdout=open(LOG, "a"), stderr=subprocess.STDOUT, env=env, start_new_session=True)
    run_t0 = time.time()
    print("Đã khởi động:", " ".join(cmd))

def drive_episodes():
    n = 0
    for d in glob.glob(f"{DRIVE_BLOCKS}/blk_*"):
        if os.path.exists(f"{d}/DONE"):
            n += json.load(open(f"{d}/DONE"))["episodes"]
    return n

def running_blocks():
    """Tên các khối có generate.py đang chạy (đọc /proc)."""
    names = set()
    for f in glob.glob("/proc/[0-9]*/cmdline"):
        try:
            a = open(f, "rb").read().decode(errors="replace").split("\0")
        except OSError:
            continue
        if any(x.endswith("generate.py") for x in a) and "--out" in a:
            names.add(os.path.basename(a[a.index("--out") + 1]))
    return names

def block_progress():
    """Các khối đang chạy: chuỗi "khối:đạt/thử" và tổng số episode đạt chưa lên Drive."""
    out, inflight = [], 0
    for name in sorted(running_blocks()):
        f = f"/content/work/logs/{name}.log"
        lines = open(f, errors="replace").read().splitlines() if os.path.exists(f) else []
        ok = sum("SUCCESS" in l for l in lines); tried = sum(l.startswith("[seed") for l in lines)
        inflight += ok
        out.append(f"{name[4:]}:{ok}/{tried}")
    return " ".join(out[-12:]), inflight

def now():                          # giờ Việt Nam (Colab chạy theo UTC)
    return datetime.now(timezone(timedelta(hours=7)))

# ETA: tốc độ = số episode đạt (đã lên Drive + đang nằm trong khối chưa xong) kể từ lúc khởi động run_blocks.py
if "run_n0" not in globals() or globals().get("run_n0_t0") != run_t0:
    run_n0, run_n0_t0 = drive_episodes(), run_t0
seen = globals().get("seen", 0)   # chạy lại cell: không in lại log đã in
while run_proc.poll() is None:
    time.sleep(60)
    n = drive_episodes()
    blocks, inflight = block_progress()
    made, hours = n - run_n0 + inflight, (time.time() - run_t0) / 3600
    rate = made / max(hours, 1e-6)
    left = max(TARGET - n - inflight, 0)
    if made >= 3:
        eta_h = left / rate
        eta = f"còn ~{eta_h:.1f} giờ, xong lúc ~{(now() + timedelta(hours=eta_h)).strftime('%H:%M %d/%m')}"
    else:
        eta = "ETA: chờ đủ 3 episode đạt"
    log = open(LOG, errors="replace").read().splitlines()
    for l in log[seen:]:            # khối xong / khối lỗi (in nguyên văn, kể cả traceback)
        if not re.match(r"^\d\d:\d\d \d+/", l):
            print("   ", l)
    seen = len(log)
    print(f"{now():%H:%M} | Drive: {n}/{TARGET} (+{inflight} đang chạy) | {rate:.0f} episode/giờ | {eta} | "
          f"khối (đạt/thử): {blocks}", flush=True)
print("Xong, mã thoát", run_proc.returncode, "| Drive:", drive_episodes(), "episode")
print("\n".join(open(LOG, errors="replace").read().splitlines()[-20:]))


In [ ]:
# [6] Xem log và tỉ lệ đạt của các khối (chạy lúc nào cũng được)
import glob, os, re
print("".join(open("/content/work/run_blocks.log").readlines()[-15:]))
ok = tried = 0
for f in glob.glob(f"{DRIVE_BLOCKS}/blk_*/generate.log"):
    s = open(f).read(); ok += s.count("SUCCESS"); tried += len(re.findall(r"^\[seed", s, re.M))
print(f"Các khối trên Drive: {ok} đạt / {tried} lần thử ({100 * ok / max(tried, 1):.0f}%)")

In [ ]:
# [7] Dừng sinh dữ liệu (các khối đang chạy dở sẽ mất, các khối đã lên Drive vẫn giữ)
import os, signal
if "run_proc" in globals() and run_proc.poll() is None:
    os.killpg(os.getpgid(run_proc.pid), signal.SIGTERM)
    print("Đã dừng.")